# B1：把画面压成 token，再预测下一帧

这一份 Notebook 只增加两个困难：先把图片变短，再让模型预测下一个离散表示。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.video import TinyVQVAE, ActionTokenTransformer, video_batch_from_episodes, token_accuracy
torch.manual_seed(0)


## 1. 先把时间接对

第 `t` 个按键造成 `frame[t] → frame[t+1]`。把动作错开一格，模型会学到一套看似收敛却无法控制的规律。

In [ ]:
episodes = make_pixelworld_dataset(num_episodes=8, length=8, seed=0)
current, actions, following = video_batch_from_episodes(episodes)
print('current/action/following:', tuple(current.shape), tuple(actions.shape), tuple(following.shape))
assert len(current) == len(actions) == len(following)


## 2. 先预热普通 Autoencoder

若一开始同时学习视觉特征、码本和 Decoder，小数据很容易让所有位置只用一个码字。先学会粗略重建，再把已有特征分配给码本。

In [ ]:
images = torch.cat((current, following))
tokenizer = TinyVQVAE(codebook_size=16, embedding_size=8)
optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)
for _ in range(30):
    optimizer.zero_grad(); loss, _ = tokenizer.continuous_loss(images); loss.backward(); optimizer.step()
tokenizer.initialize_codebook(images)
print('AE warm-up loss:', round(float(loss.detach()), 4))


## 3. 打开 VQ 与 STE

前向真的选择最近码字；反向用 STE 把 Decoder 的梯度近似送回 Encoder。这里同时检查重建和码本使用数。

In [ ]:
optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-4)
losses = []
for _ in range(20):
    optimizer.zero_grad(); output = tokenizer(images); output['loss'].backward(); optimizer.step(); losses.append(float(output['loss'].detach()))
used_codes = torch.unique(output['tokens']).numel()
print('VQ loss:', round(losses[0], 4), '→', round(losses[-1], 4), 'used codes:', used_codes)
assert used_codes > 1


## 4. 用当前 token 与动作预测下一 token

一张 `16×16` 图片现在只剩 `4×4=16` 个编号。动作 embedding 会加到每个位置，模型才能区分向左和向右。

In [ ]:
with torch.no_grad():
    current_tokens = tokenizer.encode_tokens(current).flatten(1)
    next_tokens = tokenizer.encode_tokens(following).flatten(1)
dynamics = ActionTokenTransformer(codebook_size=16, model_size=32)
optimizer = torch.optim.Adam(dynamics.parameters(), lr=3e-3)
losses = []
for _ in range(35):
    optimizer.zero_grad(); loss = dynamics.loss(current_tokens, actions, next_tokens); loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
accuracy = token_accuracy(dynamics(current_tokens, actions), next_tokens)
print('token loss:', round(losses[0], 3), '→', round(losses[-1], 3), 'accuracy:', round(float(accuracy.detach()), 3))
assert losses[-1] < losses[0]


## 小结

我们完成了 `frame → VQ token → action-conditioned next token`。训练准确率只说明这批小数据可拟合；B2 会固定同一画面更换动作，并让模型连续吃自己的输出。